# 🎓 金融硕士学位论文 - 实证分析标准化工作台与目录大纲模板

本模板专为**金融硕士（MF）学位论文**及**经管类实证学术研究**设计。目录结构与实证流程完全对标国内顶级期刊（如《经济研究》、《管理世界》、《金融研究》）的严谨范式。

**💡 使用说明**：
- **大标题目录**：已固定学术论文实证部分的标准框架。
- **内容指南**：每个章节下方标注了**“学术惯例与写作要点”**，说明该步骤在论文中需要写什么、怎么写。
- **代码框架**：各章节配备了对应的 Python (Pandas/DuckDB/Linearmodels) 代码脚手架，可直接代入数据运行。

---

## 📁 目录大纲与实证分析全景图
1. **第一部分：样本筛选与数据准备 (Data Selection & Data Preparation)**
   - 1.1 数据来源与初始样本界定
   - 1.2 样本清洗与剔除标准
   - 1.3 关键变量定义与测度
   - 1.4 学术缩尾处理 (Winsorize)
2. **第二部分：变量定义与计量模型设定 (Variable Definition & Model Specification)**
   - 2.1 计量模型设定与公式规范
   - 2.2 变量定义表 (Table 1)
3. **第三部分：描述性统计与相关性矩阵 (Descriptive Stats & Correlation Matrix)**
   - 3.1 主要变量描述性统计 (Table 2)
   - 3.2 变量间相关系数矩阵 (Table 3 - Pearson & Spearman)
   - 3.3 多重共线性检验 (VIF 检验)
4. **第四部分：基准回归分析 (Baseline Regression Analysis)**
   - 4.1 面板双固定效应 (Two-way Fixed Effects) 基准回归结果 (Table 4)
   - 4.2 回归结果的统计与经济显著性阐释
5. **第五部分：内生性检验与处理 (Endogeneity Mitigation)**
   - 5.1 工具变量法 (2SLS / IV Approach)
   - 5.2 倾向得分匹配法 (PSM - Propensity Score Matching)
   - 5.3 核心自变量滞后一期回归 (Lagged Explanatory Variables)
   - 5.4 差分中差分法 (DID) 或 准自然实验
6. **第六部分：稳健性检验 (Robustness Tests)**
   - 6.1 替换因变量/自变量测度指标
   - 6.2 改变样本选择窗口 / 重新筛选子样本
   - 6.3 改变聚类标准误层级 (Clustered Standard Errors)
   - 6.4 安慰剂检验 (Placebo Test)
7. **第七部分：机制分析与渠道检验 (Mechanism Analysis / Channel Test)**
   - 7.1 中介效应分析 (Mediation Effect - 三步法 / 新式机制检验)
   - 7.2 调节效应与交互项分析 (Moderation Effect / Interaction Term)
8. **第八部分：异质性分析 (Heterogeneity Analysis)**
   - 8.1 产权性质异质性 (SOE vs. Non-SOE)
   - 8.2 企业规模/融资约束异质性
   - 8.3 行业门类/地区市场化进程异质性

In [ ]:
# 0. 环境准备：导入金融实证分析核心依赖库
import os
import duckdb
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ 实证分析核心库导入成功！")

# 一、 样本筛选与数据准备 (Data Selection & Data Preparation)

### 1.1 学术惯例与写作要点：
- **数据来源说明**：必须明确阐述各大数据库的名称。例如：上市公司财务数据来自 **CSMAR** 或 **Wind**，公司治理数据来自 **CNRDS**，地方统计数据来自《中国城市统计年鉴》等。
- **样本筛选标准（金融论文核心步骤）**：在论文正文中，必须详细罗列以下剔除标准，以向评审展示样本的纯净性：
  1. **剔除金融、保险及证券行业公司**：由于金融类企业的资产负债结构和业务特征极度特殊，其财务指标与其他行业不具可比性，因此必须剔除（通常指证监会2012大类J门类行业）；
  2. **剔除 ST、*ST 及 PT 等交易状态异常的企业**：这类企业面临退市风险、财务异常或有重大重组，其财务数据存在极度异常值；
  3. **剔除核心变量存在缺失值的样本**：保证回归样本的连贯性和一致性；
  4. **根据研究需要进行的特殊剔除**：例如剔除上市未满一年的企业、剔除资产负债率大于 100% 的资不抵债企业等。
- **缩尾处理 (Winsorize)**：为了避免极端值（Outliers）干扰，实证论文必须在变量定义后声明：“为了消除极端值对回归结论的影响，我们对所有连续变量进行了双侧 1%（或 5%）的 Winsorize 缩尾处理”。

In [ ]:
# 1. 数据准备与样本清洗核心代码模板
db_path = "../data/warehouse/financial_warehouse.db"
if not os.path.exists(db_path):
    print("⚠️ 提示：未在 ../data/warehouse/ 下找到 DuckDB 文件，请检查路径。")
else:
    conn = duckdb.connect(db_path)
    # 从 DWS 层拉取面板数据并进行初步清洗（变量均为数仓真实存在的财务指标）
    sql_filter = """
        SELECT stkcd, company_name, year, industry_category,
               firm_size, leverage, roa, roe, revenue_growth,
               tobin_q, lag_roa, assets, cash
        FROM dws.firm_year_panel
        WHERE industry_category != '金融业'  -- 剔除金融业（实证惯例）
    """
    df_raw = conn.execute(sql_filter).fetchdf()
    conn.close()

    # 经典公司金融变量：现金持有水平 = 货币资金 / 总资产
    df_raw['cash_holdings'] = df_raw['cash'] / df_raw['assets']

    # 双侧 1% 缩尾处理 (Winsorization) 演示
    df_clean = df_raw.copy()
    vars_to_winsor = ['firm_size', 'leverage', 'roa', 'roe', 'revenue_growth',
                      'tobin_q', 'lag_roa', 'cash_holdings']
    for var in vars_to_winsor:
        if var in df_clean.columns:
            q_low = df_clean[var].quantile(0.01)
            q_high = df_clean[var].quantile(0.99)
            df_clean[var] = df_clean[var].clip(q_low, q_high)


# 二、 变量定义与计量模型设定 (Variable & Model Specification)

### 2.1 学术惯例与写作要点：
- **变量定义表 (Table 1: Variable Definition)**：
  在论文中以三线表形式呈现，包含：
  - **变量类型**：被解释变量（Dependent Variable）、解释变量（Independent Variable）、控制变量（Control Variables）、固定效应项（Fixed Effects）；
  - **变量符号**：如 $Y$, $X$, $Size$, $Lev$, $ROA$ 等；
  - **变量名称**：如 融资约束、现金持有、企业规模、资产负债率 等；
  - **计算方法与定义说明**：如“期末总资产的自然对数”、“期末负债总额 / 期末资产总额”等。
- **基准模型公式规范**：
  必须在论文中写出标准的面板数据计量模型公式。例如以双固定效应为例：
  $$Y_{i,t} = \beta_0 + \beta_1 X_{i,t} + \sum_{k} \gamma_k Control_{k,i,t} + \mu_i + \lambda_t + \varepsilon_{i,t}$$
  - $i$ 代表个体（企业），$t$ 代表时间（年度）；
  - $Y_{i,t}$ 为被解释变量， $X_{i,t}$ 为核心解释变量；
  - $Control_{k,i,t}$ 为一系列控制变量的集合，$\gamma_k$ 为对应系数；
  - $\mu_i$ 为企业个体固定效应（Individual Fixed Effect），控制企业层面不随时间变化的遗漏变量（如企业文化、初始产权性质）；
  - $\lambda_t$ 为年度固定效应（Year Fixed Effect），控制时间维度上所有企业共同面临的宏观经济冲击（如财政货币政策、宏观经济周期）；
  - $\varepsilon_{i,t}$ 为随机扰动项。

# 三、 描述性统计与相关性分析 (Descriptive Stats & Correlation)

### 3.1 学术惯例与写作要点：
- **主要变量描述性统计 (Table 2)**：
  - **包含指标**：变量名称、样本量 ($N$)、平均值 ($Mean$)、标准差 ($S.D.$)、最小值 ($Min$)、中位数 ($Median$)、最大值 ($Max$)。
  - **学术分析重点**：对比核心变量的均值和标准差，说明样本内企业在关键特征上是否存在显著的离散性；对比最大值与最小值，展示缩尾后的变动幅度，确保数据中没有未处理的极大异常值。
- **相关系数矩阵 (Table 3)**：
  - **Pearson 相关系数**：展示在表格的下方对角线（Lower Triangle）。一般若核心自变量与因变量的相关系数显著且符号符合预期，说明基准结论大概率成立。
  - **Spearman 相关系数**（非必做，有些展示在上方对角线）：用于检查非线性单调关系。
  - **共线性判断标准**：各控制变量之间的相关系数如果绝对值超过 **0.7**（也有文献为 0.8），说明可能存在严重的多重共线性风险，需要关注。
- **多重共线性检验 (VIF 检验)**：
  - 通常在描述性统计或相关性分析后提及：“为了进一步排除多重共线性的影响，我们对模型进行了 VIF 检验，结果显示各变量的 VIF 均显著小于 10（或 5），表明模型不存在严重的多重共线性问题”。

In [ ]:
# 2. 描述性统计、相关性矩阵及 VIF 检验代码脚手架
if 'df_clean' in locals():
    # (1) 描述性统计
    desc_table = df_clean[vars_to_winsor].describe().T[['count', 'mean', 'std', 'min', '50%', 'max']]
    desc_table.columns = ['N', 'Mean', 'S.D.', 'Min', 'Median', 'Max']
    print("📊 Table 2: Descriptive Statistics 描述性统计表")
    display(desc_table.round(4))
    
    # (2) Pearson 相关系数矩阵
    corr_table = df_clean[vars_to_winsor].corr(method='pearson')
    print("\n🔗 Table 3: Pearson Correlation Matrix 相关系数矩阵")
    display(corr_table.round(4))
    
    # (3) VIF 共线性检验
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    vif_df = df_clean[vars_to_winsor].dropna()
    vif_data = pd.DataFrame()
    vif_data["Variable"] = vif_df.columns
    vif_data["VIF"] = [variance_inflation_factor(vif_df.values, i) for i in range(vif_df.shape[1])]
    print("\n📐 VIF Multicollinearity Test VIF 共线性检验报告")
    display(vif_data.round(2))

# 四、 基准回归分析 (Baseline Regression Analysis)

### 4.1 学术惯例与写作要点：
- **渐进式结果汇报 (Table 4)**：
  优秀的金融/财务论文，基准回归表绝对不应该只放一列最终结果，而是要呈现**由浅入深的渐进过程**：
  - **第 (1) 列**：只放入核心自变量，不加入控制变量，不控制固定效应；
  - **第 (2) 列**：加入核心自变量和所有控制变量，但不控制固定效应；
  - **第 (3) 列**：控制控制变量，并控制行业（或个体）固定效应；
  - **第 (4) 列**：终极双固定效应模型，加入所有控制变量，同时控制**个体固定效应**和**年度固定效应**（最经典规范的模型）。
- **标准误聚类处理 (Clustered Standard Errors)**：
  必须在表格下方加注：“括号内为在企业层面对齐聚类调整后的 $t$ 值”。这是因为单机 ODS 常规 OLS 标准误会由于时间序列的自相关被严重低估，从而导致过度显著。聚类到企业层面（Clustered by Firm/Stkcd）是金融公司治理研究中不可动摇的底线。
- **系数解读逻辑**：
  1. **符号与方向**：系数 $\beta_1$ 的正负号是否符合理论假说；
  2. **统计显著性**：是否带星号（*** 代表 1% 显著，** 代表 5% 显著，* 代表 10% 显著）；
  3. **经济显著性**：除了统计上的 p 值外，还要做经济含义阐释。例如“解释变量每增加一个标准差，被解释变量将随之变动 $\beta_1 \times S.D.(X) / S.D.(Y)$ 个标准差，说明其经济效应具有现实意义”。

In [ ]:
# 3. 面板双固定效应 (Clustered Standard Errors) 基准回归脚手架
if 'df_clean' in locals():
    # 准备面板所需的多级索引 DataFrame
    reg_df = df_clean.dropna().copy()
    reg_df['firm_id'] = reg_df['stkcd'].astype('category').cat.codes
    reg_df = reg_df.set_index(['firm_id', 'year'])
    
    # 常数项
    reg_df['const'] = 1.0
    
    # 设定因变量 (Dependent Variable) 与自变量组合
    # 示例自变量：cash_holdings，控制变量：leverage, tobin_q, roa
    exog_vars = ['const', 'cash_holdings', 'leverage', 'tobin_q', 'roa']
    endog_var = reg_df['roa']
    
    # 运行双向固定效应模型 (Entity Effects + Time Effects)
    model = PanelOLS(endog_var, reg_df[exog_vars], entity_effects=True, time_effects=True)
    # 在企业(entity)层面聚类标准误
    results = model.fit(cov_type='clustered', cluster_entity=True)
    
    print("📝 Table 4: Baseline Regression Results 基准面板双固定效应回归结果")
    print(results.summary)

# 五、 内生性检验与处理 (Endogeneity Mitigation)

### 5.1 学术惯例与写作要点：
- **内生性来源剖析（必须写明）**：
  在实证回归中，结论可能受到内生性干扰。必须在论文中阐述可能的内生性原因：
  1. **反向因果 (Reverse Causality)**：解释变量影响被解释变量的同时，被解释变量也可能反过来决定解释变量的取值；
  2. **遗漏变量偏差 (Omitted Variable Bias)**：基准回归无法控制所有随时间改变的非观测企业特征或宏观变量；
  3. **测量误差 (Measurement Error)**：部分学术概念难以用财务指标精确度量。
- **学术常用的几种内生性解决方法（论文需要选择至少 1-2 种进行严谨应对）**：
  1. **工具变量法 (IV - 2SLS)**：寻找满足“**相关性**”（与核心自变量强相关）与“**外生性**”（与扰动项不相关，只通过影响自变量进而影响因变量）的工具变量。例如使用**同行/同地区均值**作为工具变量；运行第一阶段和第二阶段回归，汇报不可识别检验（Kleibergen-Paap LM 统计量）、弱工具变量检验（Cragg-Donald Wald F 统计量）及过度识别检验；
  2. **倾向得分匹配法 (PSM)**：如果解释变量是二分虚拟变量（或可以通过中位数划分为0/1组），计算倾向得分进行匹配。可以消除控制变量在处理组和控制组之间的结构性非平衡性，之后用匹配后样本重新进行基准回归；
  3. **自变量滞后一期回归 (Lagged Explanatory Variable)**：把被解释变量置于 $t$ 期，核心自变量和所有控制变量全部滞后一期（采用 $t-1$ 期数据），这是缓解反向因果最简便有效的学术做法；
  4. **准自然实验与双重差分法 (DID)**：如果有外生的政策试点，构建 DID/多时点 DID 模型，汇报平行趋势检验结果（金融政策/宏观政策实证论文的首选）。

In [ ]:
# 4. 工具变量两阶段最小二乘法 (IV-2SLS) 示例框架
from linearmodels.iv import IV2SLS

# 提示：2SLS 计量模型示例：Y = const + Controls + (X ~ IV)
# 示例公式：kz_index ~ const + leverage + tobin_q + [cash_holdings ~ industry_avg_cash]
# 在此处展示如何调用 IV2SLS 脚手架

print("💡 工具变量法 (2SLS) 调库脚手架：")
print("""
from linearmodels.iv import IV2SLS
# 实例化 2SLS 模型 (注意：此处需要你的数据集内真实包含工具变量 IV_column)
# iv_model = IV2SLS(dependent=reg_df['kz_index'], 
#                   exog=reg_df[['const', 'leverage', 'tobin_q']], 
#                   endog=reg_df['cash_holdings'], 
#                   instruments=reg_df['IV_column'])
# iv_results = iv_model.fit(cov_type='clustered')
# print(iv_results.summary)
""")

# 六、 稳健性检验 (Robustness Tests)

### 6.1 学术惯例与写作要点：
- **稳健性检验是硕士论文不可或缺的“压舱石”**。用以证明前文的研究结论并不是在特定的参数设置或特殊的样本选择下偶然形成的。一般来说，必须汇报 **3 种或更多** 稳健性检验：
  1. **替换因变量测度**：改变被解释变量的度量指标（如将 $ROA$ 换成 $ROE$ 或净利润，将融资约束 $KZ$ 指数换成 $SA$ 指数）；
  2. **替换解释变量测度**：改变核心自变量的度量方式；
  3. **改变样本选择窗口**：例如剔除某些特殊年度（如2020年新冠疫情冲击年），或者仅选择特定行业大类的子样本；
  4. **改变聚类标准误层级**：如把原来在“企业”层面的聚类标准误，升级调整为“行业-年份”双向聚类，或者“省份-年份”聚类，证明即便提高标准误门槛，星号仍然显著；
  5. **安慰剂检验 (Placebo Test)**：
     - **经典做法**：随机化样本的处理状态（如果是 DID）或者**随机打乱核心自变量的配对顺序**，重复模拟回归多达 500 次或 1000 次。
     - **汇报图**：将这 1000 次伪随机回归得到的回归系数和 p 值绘制成核密度分布图。若发现随机化后的系数均值集中在 0 附近且绝大多数不显著，而论文中真实的基准回归系数（以红线标出）落在分布图的尾端（显著异于 0），则极有说服力地证明了原结论具有强大的稳健性。

In [ ]:
# 5. 安慰剂检验 (打乱自变量顺序以构造随机伪系数分布) 运行模板
if 'reg_df' in locals():
    coef_simulations = []
    print("🧪 安慰剂模拟进行中 (蒙特卡洛随机模拟 200 次)...")
    
    for i in range(200):
        # 随机打乱自变量 cash_holdings 的顺序，彻底打断其与因变量的经济联系
        shuffled_x = reg_df['cash_holdings'].sample(frac=1).values
        reg_df['shuffled_x'] = shuffled_x
        
        # 跑一个简化的面板回归
        temp_model = PanelOLS(reg_df['roa'], reg_df[['const', 'shuffled_x', 'leverage']], entity_effects=True)
        temp_results = temp_model.fit()
        coef_simulations.append(temp_results.params['shuffled_x'])
        
    # 绘制模拟系数的核密度图，并标出真实回归系数（假设基准中 cash_holdings 的真实系数为 -0.32）
    plt.figure(figsize=(8, 5))
    sns.kdeplot(coef_simulations, fill=True, color='#475569', label='随机模拟系数分布')
    plt.axvline(x=-0.32, color='red', linestyle='--', linewidth=2, label='基准回归真实系数 (True Coef = -0.32)')
    plt.axvline(x=0.0, color='black', linestyle=':', linewidth=1)
    plt.title('Placebo Test Plot (安慰剂检验系数核密度分布图)')
    plt.xlabel('Estimated Coefficient')
    plt.ylabel('Density')
    plt.legend(loc='upper right')
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.show()

# 七、 机制分析与渠道检验 (Mechanism Analysis & Channel Test)

### 7.1 学术惯例与写作要点：
- **为什么要做机制检验？**：
  基准回归只回答了“$X$ 是否能显著影响 $Y$”，而评审专家和答辩委员最关心的是**“$X$ 是通过什么科学渠道或路径作用于 $Y$ 的”**。不探讨作用机制的论文，在学术深度上是单薄的。
- **机制分析的两大主线流程**：
  1. **中介效应检验 (Mediation Effect)**：
     - **传统的温忠麟三步法**：
       - 第一步：验证 $X$ 对 $Y$ 的总效应是否显著；
       - 第二步：验证 $X$ 对中介变量 $M$ 的效应是否显著；
       - 第三步：把 $X$ 和 $M$ 同时放入模型对 $Y$ 进行回归，观察 $M$ 是否显著，且 $X$ 的系数是否变小（部分中介）或不显著（完全中介）。
     - **新式机制检验**（目前《经济研究》等期刊更提倡）：由于三步法存在统计学上的过度识别或逻辑循环批评，现在主流做法是**机制渠道的经济逻辑论证**，即只需汇报第二步（自变量 $X$ 对中介路径 $M$ 的促进或抑制作用）并做深度理论阐释即可。
  2. **调节效应检验 (Moderation Effect / Interaction Term)**：
     - 通过在计量模型中引入**“自变量 × 调节变量”的交互项（Interaction Term, 即 $X \times W$）**。
     - **调节效应显著判定**：观察交互项的系数是否显著。如果交互项系数显著，说明在不同特征的外部环境（如市场化进程、产权性质）或内部特征（如企业杠杆率、管理层能力）作用下，$X$ 对 $Y$ 的边际促进或抑制效应会发生显著的增强或减弱。

In [ ]:
# 6. 机制与调节效应：交互项回归模板
if 'reg_df' in locals():
    # 以 leverage 作为调节变量为例，构建核心解释变量与调节变量的乘积项
    reg_df['interaction'] = reg_df['cash_holdings'] * reg_df['leverage']
    
    # 设定包含常数项、核心自变量、调节变量、交互项以及其他控制变量的解释变量列表
    exog_mech = ['const', 'cash_holdings', 'leverage', 'interaction', 'tobin_q', 'roa']
    
    # 运行包含交互项的面板回归
    mech_model = PanelOLS(reg_df['roa'], reg_df[exog_mech], entity_effects=True, time_effects=True)
    mech_results = mech_model.fit(cov_type='clustered', cluster_entity=True)
    
    print("📝 Table 5: Moderation Effect Results 调节效应回归结果（交互项）")
    print(mech_results.summary)

# 八、 异质性分析 (Heterogeneity Analysis)

### 8.1 学术惯例与写作要点：
- **异质性分析（又称敏感性分析/分组回归）**：
  用以探讨核心结论在不同性质的样本组中是否存在非对称性。这是丰富实证结论、增强政策建议具体性的核心段落。
- **金融与公司财务研究中最经典的 3 大异质性分组维度**：
  1. **产权性质异质性 (SOE vs. Non-SOE)**：
     - **国企与非国企**在融资约束、软预算约束、代理成本、社会责任等方面有天壤之别，因此几乎是所有中国公司金融论文的必做分组；
  2. **企业规模或融资约束水平异质性**：
     - 例如按企业资产中位数划分为“大型企业”与“小型企业”，或者“高融资约束”与“低融资约束”组；
  3. **外部环境与地理区域异质性**：
     - 如东部地区 vs. 中西部地区企业，或者按樊纲“市场化指数”划分为高、低市场化地区。
- **实证写作规范**：
  - 将样本按维度一分为二（或一分为三），分别进行基准双固定效应回归，呈现为两列并列的表格。
  - 对比两组自变量系数的**大小、符号以及显著性**的差异，并提供深刻的中国制度背景下的经济学机理解释。
  - **系数差异检验（Chow 检验/Bootstrap）**：严格学术规范要求在分组回归后进行“组间系数差异检验”，证明两组的系数在统计学上确实存在显著差异，而不仅仅是“一组显著，另一组不显著”的表面现象。

In [ ]:
# 7. 异质性分析分组回归框架示意代码
print("💡 异质性分析（分组回归）标准实现逻辑：")
print("""
# 假设数据集中已构建产权分类 dummy 字段: is_soe (1代表国企, 0代表非国企)
# soe_sample = reg_df[reg_df['is_soe'] == 1]
# non_soe_sample = reg_df[reg_df['is_soe'] == 0]

# 运行国企组回归
# model_soe = PanelOLS(soe_sample['kz_index'], soe_sample[exog_vars], entity_effects=True, time_effects=True)
# res_soe = model_soe.fit(cov_type='clustered', cluster_entity=True)

# 运行非国企组回归
# model_non_soe = PanelOLS(non_soe_sample['kz_index'], non_soe_sample[exog_vars], entity_effects=True, time_effects=True)
# res_non_soe = model_non_soe.fit(cov_type='clustered', cluster_entity=True)

# 并在 LaTeX 表格中将两组结果并列汇报，进行系数对比与组间 Chow 检验说明
""")